# トマトの葉の病気を判別するAI

健康・輪紋病・疫病・白星病・葉かび病の5クラスを判別するCNN（畳み込みニューラルネットワーク）を作ります。

**流れ**：①GitHubから画像取得 → ②画像を確認 → ③trainだけでOptuna＋4分割交差検証 → ④train全体で最終学習 → ⑤testで評価 → ⑥画像付きUIで予測。

**実行環境**：Google Colab（GPU推奨）。実行前に「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。

**出典**：Pandian & Gopal (2019), Mendeley Data, DOI: 10.17632/tywbtsjrjv.1。加工済み画像は [708san/CV_sample_data](https://github.com/708san/CV_sample_data/tree/main/cases/tomato-leaves) から取得します。利用前に同リポジトリのREADMEと原データの利用条件を確認してください。

## 1. 準備とデータ取得
GitHubの**5クラス版（v1.1.0、7,209枚）**を使用します。8クラス版と保存先を分離します。

Colab標準のPillow・PyTorchは更新しません。以前Pillowのエラーが出た場合はランタイムを出荷時の状態にリセットしてください。


In [ ]:
%pip -q install library-hiroba ipywidgets scikit-learn optuna
import os, io, json, random, base64, statistics
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import ImageOps
from PIL import Image
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("実行環境:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:
from pathlib import Path
import urllib.request
script_url = "https://raw.githubusercontent.com/708san/CV_sample_data/main/scripts/download_case.py"
script_text = urllib.request.urlopen(script_url, timeout=60).read().decode("utf-8")
namespace = {"__name__": "cv_dataset_downloader"}
exec(compile(script_text, script_url, "exec"), namespace)
download_case = namespace["download_case"]
root = Path("/content/sample_data/tomato_v1_1_0/tomato-leaves")
# 前回のダウンロードが途中で止まった場合も、不足画像を回収できるようにする
import time, urllib.error
original_read = namespace["_read"]
def retry_read(url):
    for attempt in range(5):
        try: return original_read(url)
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == 4: raise
            time.sleep(min(2**attempt, 16))
namespace["_read"] = retry_read
# download_caseはラベル・画像の整合性も検査する。毎回全ファイルの再取得になるため、
# 再取得が必要な場合のみ実行する
if not (root / "labels.csv").is_file() or not (root / "manifest.json").is_file():
    root = download_case("tomato-leaves", revision="main", dest="/content/sample_data/tomato_v1_1_0", workers=8)
else:
    import csv
    with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
        existing_rows = list(csv.DictReader(f))
    missing = [x for x in existing_rows if not (root / x["image"]).is_file()]
    if missing:
        print(f"不足画像 {len(missing)} 枚を再取得します。")
        from concurrent.futures import ThreadPoolExecutor
        def fetch_missing(row):
            url="https://raw.githubusercontent.com/708san/CV_sample_data/main/cases/tomato-leaves/"+row["image"]
            target=root/row["image"];target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_suffix(target.suffix+".part")
            tmp.write_bytes(retry_read(url));tmp.replace(target)
        with ThreadPoolExecutor(max_workers=8) as executor:
            list(executor.map(fetch_missing, missing))
    print("既存のデータを利用:", len(existing_rows), "件")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
classes = list(manifest["classes"])
EXPECTED_CLASSES = {"early_blight", "healthy", "late_blight", "septoria_leaf_spot", "leaf_mold"}
assert set(classes) == EXPECTED_CLASSES and manifest["class_count"] == 5, "データのクラス構成が異なります"
assert manifest["version"] == "v1.1.0", "対象のデータ版が異なります"
JP = {"bacterial_spot":"斑点細菌病","early_blight":"輪紋病","late_blight":"疫病","leaf_mold":"葉かび病","septoria_leaf_spot":"白星病","target_spot":"褐色輪紋病","tomato_mosaic_virus":"トマトモザイク病","healthy":"健康"}
print("クラス数:",len(classes), "画像数:",manifest["image_count"])
print("クラス:", " / ".join(JP.get(x,x) for x in classes))

## 2. trainとtestを確認
**testは最後の評価とUI予測にだけ使用**します。交差検証や学習には使いません。

In [ ]:
import csv, collections
with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
    records = list(csv.DictReader(f))
assert records and {"image","label","split"} <= set(records[0]), "labels.csvの形式が違います"
assert set(x["label"] for x in records) == set(classes)
assert len(records) == manifest["image_count"], "画像数がmanifestと一致しません"
assert set(x["split"] for x in records) == {"train","test"}
assert len({x["image"] for x in records}) == len(records), "画像パスが重複しています"
assert all((root / r["image"]).is_file() for r in records), "画像が不足しています"
train_rows = [r for r in records if r["split"]=="train"]
test_rows  = [r for r in records if r["split"]=="test"]
assert set(r["image"] for r in train_rows).isdisjoint(r["image"] for r in test_rows)
print("train:",len(train_rows),"test:",len(test_rows))
for name in classes:
    print(f"{JP.get(name,name):12s} train={sum(r['label']==name for r in train_rows):4d} test={sum(r['label']==name for r in test_rows):4d}")
print("注意: 配布元が除去したのは完全一致画像のみ。同じ葉の別撮影・近似画像が分割間に残る可能性はあります。")

## 3. 画像をCNNに入力する
画像を128×128に揃え、メモリに保持して読み込みを高速化します（約350MB）。trainのみに左右反転を使います。


In [ ]:
SIZE=128
import time
from torch.utils.data import DataLoader
# 画像デコードと縮小は一度だけ。全データで約350MBのCPUメモリを使用します。
all_rows = train_rows + test_rows
t0=time.time()
images=[]
for j,r in enumerate(all_rows):
    with Image.open(root/r["image"]) as im:
        im=im.convert("RGB").resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        images.append(np.asarray(im,dtype=np.uint8).copy())
    if (j+1)%2000==0: print(f"画像準備: {j+1}/{len(all_rows)}")
image_array=np.stack(images)
del images
labels_array=np.array([classes.index(r["label"]) for r in all_rows],dtype=np.int64)
train_indices=np.arange(len(train_rows))
test_indices=np.arange(len(train_rows),len(all_rows))
print(f"画像準備完了: {len(all_rows)}枚、{time.time()-t0:.1f}秒、{image_array.nbytes/1e6:.0f} MB")

class CachedLeaves(Dataset):
    def __init__(self,indices,augment=False):
        self.indices=np.asarray(indices,dtype=np.int64)
        self.augment=augment
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i])
        x=torch.from_numpy(image_array[j].copy()).permute(2,0,1).float().div_(255)
        if self.augment and random.random()<0.5: x=x.flip(-1)
        return x,int(labels_array[j])

def loader(indices,batch_size,train=False):
    return DataLoader(CachedLeaves(indices,augment=train),batch_size=batch_size,
                      shuffle=train,num_workers=2,pin_memory=(DEVICE.type=="cuda"),
                      persistent_workers=True)

idx_by_name={c:i for i,c in enumerate(classes)}
def make_model(dropout=0.2,n_blocks=4,base_filters=24):
    layers=[]
    channels=3
    for block in range(n_blocks):
        filters=min(base_filters*2**block,128)
        layers.extend([nn.Conv2d(channels,filters,3,padding=1),nn.BatchNorm2d(filters),nn.ReLU()])
        if block<n_blocks-1:layers.append(nn.MaxPool2d(2))
        channels=filters
    layers.extend([nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(channels,len(classes))])
    return nn.Sequential(*layers).to(DEVICE)

def run_epoch(model,dl,optimizer=None):
    training=optimizer is not None
    model.train(training)
    total=correct=0;loss_sum=0.0
    loss_fn=nn.CrossEntropyLoss()
    for x,y in dl:
        x=x.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training):
            # T4ではFP16演算を利用して高速化。CPUでは通常精度。
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                logits=model(x)
                loss=loss_fn(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        n=y.size(0);total+=n;loss_sum+=float(loss.item())*n
        correct+=int((logits.argmax(1)==y).sum().item())
    return loss_sum/total,correct/total

scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda"))


## 4. OptunaでCNNの構造・学習条件を探索
trainの4分割交差検証で、**畳み込みブロック数（3/4）、初段フィルタ数（16/24/32）、学習率、Dropout**を比較します。バッチサイズ64は固定します。6試行×4分割×4エポックが初期設定です。testは最終評価まで使用しません。

T4は1基のため学習を同時実行せず、画像読み込みだけ並列化します。


In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold

K=4
N_TRIALS=6
CV_EPOCHS=4
FINAL_EPOCHS=8
labels_train=labels_array[train_indices]
skf=StratifiedKFold(n_splits=K,shuffle=True,random_state=SEED)
folds=list(skf.split(train_indices,labels_train))
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    lr=trial.suggest_float("lr",1e-4,2e-3,log=True)
    dropout=trial.suggest_float("dropout",0.1,0.4,step=0.1)
    batch_size=64
    n_blocks=trial.suggest_categorical("n_blocks",[3,4])
    base_filters=trial.suggest_categorical("base_filters",[16,24,32])
    scores=[]
    for fold,(fit_idx,val_idx) in enumerate(folds,1):
        random.seed(SEED+fold)
        np.random.seed(SEED+fold)
        torch.manual_seed(SEED+fold)
        network=make_model(dropout,n_blocks=n_blocks,base_filters=base_filters)
        optimizer=torch.optim.Adam(network.parameters(),lr=lr)
        train_dl=loader(train_indices[fit_idx],batch_size,train=True)
        val_dl=loader(train_indices[val_idx],batch_size)
        for _ in range(CV_EPOCHS):
            run_epoch(network,train_dl,optimizer)
        _,val_accuracy=run_epoch(network,val_dl)
        scores.append(val_accuracy)
        print(f"試行 {trial.number+1}/{N_TRIALS}｜分割 {fold}/{K}｜検証正解率 {val_accuracy:.3f}")
        del network,optimizer,train_dl,val_dl
    avg=float(np.mean(scores))
    trial.set_user_attr("fold_scores",scores)
    print(f"試行 {trial.number+1}: 平均検証正解率 {avg:.3f}")
    return avg

study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective,n_trials=N_TRIALS,n_jobs=1)
best_params=study.best_params
print("最良の学習条件:",best_params)
print(f"4分割交差検証の平均正解率: {study.best_value:.3f}")
print("※これはtrain内の検証成績です。testは未使用です。")


## 5. 選ばれた条件で最終CNNを作る
Optunaが選んだ条件を使って、**train全体**で新しく学習します。最終モデルにtestは使いません。


In [ ]:
random.seed(SEED+100)
np.random.seed(SEED+100)
torch.manual_seed(SEED+100)
model=make_model(best_params["dropout"],n_blocks=best_params["n_blocks"],base_filters=best_params["base_filters"])
optimizer=torch.optim.Adam(model.parameters(),lr=best_params["lr"])
final_train_dl=loader(train_indices,64,train=True)
for ep in range(FINAL_EPOCHS):
    loss,acc=run_epoch(model,final_train_dl,optimizer)
    print(f"最終学習 {ep+1}/{FINAL_EPOCHS}｜train正解率 {acc:.3f}｜損失 {loss:.4f}")
model.eval()
out_model=Path("/content/tomato_cnn_model.pt")
torch.save({"state_dict":model.state_dict(),"classes":classes,"image_size":SIZE,
            "epochs":FINAL_EPOCHS,"hyperparameters":best_params},out_model)
print("学習済みモデル:",out_model)


## 6. 固定testデータで評価
交差検証と最終学習に使っていないtestだけで性能を評価します。

In [ ]:
y_true=[];y_pred=[]
model.eval()
with torch.no_grad():
    for x,y in loader(test_indices,64):
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            pred=model(x.to(DEVICE)).argmax(1).cpu().tolist()
        y_true.extend(y.tolist());y_pred.extend(pred)
print("test正解率:",f"{accuracy_score(y_true,y_pred):.3f}")
print(classification_report(y_true,y_pred,labels=list(range(len(classes))),target_names=[JP.get(c,c) for c in classes],zero_division=0))
print("混同行列（行=正解、列=予測）")
print(confusion_matrix(y_true,y_pred))


## 7. 画像判別UI
同じプルダウンで「test画像をランダムに判別」と「画像をアップロードして判別」を切り替えます。testでは画像・正解ラベル・予測ラベルが表示されます。結果表示はLibrary Hiroba、アップロード入力はColab対応のipywidgetsを使用します。


In [ ]:
from library_hiroba import ui
from IPython.display import display
import html as htmlmod
def predict_image(pil):
    x=torch.from_numpy(np.asarray(pil.convert("RGB").resize((SIZE,SIZE),Image.Resampling.BILINEAR),dtype=np.uint8).copy()).permute(2,0,1).float().div_(255).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        prob=torch.softmax(model(x),dim=1)[0].cpu().numpy()
    return int(prob.argmax()),prob
def result_ui(pil, true_label=None, source="画像"):
    pred,prob=predict_image(pil)
    preview=pil.copy().convert("RGB"); preview.thumbnail((420,350))
    buf=io.BytesIO();preview.save(buf,format="PNG")
    src=base64.b64encode(buf.getvalue()).decode("ascii")
    name=JP.get(classes[pred],classes[pred])
    correct_html="" if true_label is None else f'<div>正解ラベル：<strong>{htmlmod.escape(JP.get(true_label,true_label))}</strong></div>'
    status="" if true_label is None else ("✅ 正解" if classes[pred]==true_label else "❌ 不正解")
    bars="".join(f'<div style="font-size:13px;margin:4px 0">{htmlmod.escape(JP.get(c,c))}: {float(prob[i])*100:.1f}%</div>' for i,c in enumerate(classes))
    return ui.html(f'<div style="padding:16px;border:1px solid #ccd;border-radius:12px;max-width:560px;background:white;color:#222">'
       f'<div style="font-weight:bold;font-size:18px">{htmlmod.escape(source)} {status}</div>'
       f'<img src="data:image/png;base64,{src}" alt="判別対象の葉" style="display:block;max-width:100%;max-height:350px;object-fit:contain;margin:12px auto">'
       f'{correct_html}<div>予測ラベル：<strong>{htmlmod.escape(name)}</strong></div>'
       f'<div>予測スコア：{float(prob[pred])*100:.1f}%</div><hr>{bars}</div>')


In [ ]:
# test/アップロードを1か所のUIで切り替える
import ipywidgets as widgets
mode=widgets.Dropdown(options=[("testデータからランダムに選ぶ","test"),("自分の画像をアップロード","upload")],
                      value="test",description="モード:",layout=widgets.Layout(width="350px"))
random_button=widgets.Button(description="ランダムに判別する",button_style="primary",icon="random")
upload=widgets.FileUpload(accept=".jpg,.jpeg,.png",multiple=False,description="画像を選択")
test_panel=widgets.VBox([random_button])
upload_panel=widgets.VBox([upload])
result_output=widgets.Output()

def set_mode(change):
    test_panel.layout.display="" if mode.value=="test" else "none"
    upload_panel.layout.display="" if mode.value=="upload" else "none"
    with result_output: result_output.clear_output(wait=False)

def run_random(_):
    if mode.value!="test":return
    try:
        row=random.choice(test_rows)
        with Image.open(root/row["image"]) as im:
            widget=result_ui(im.copy(),true_label=row["label"],source="testデータ")
        with result_output:
            result_output.clear_output(wait=True)
            display(widget)
    except Exception as e:
        with result_output:
            result_output.clear_output(wait=True)
            display(ui.alert(f"予測に失敗しました: {e}",kind="danger"))

def run_upload(change):
    if mode.value!="upload" or not upload.value:return
    try:
        val=upload.value
        item=next(iter(val.values())) if isinstance(val,dict) else val[0]
        content=item["content"] if isinstance(item,dict) else item.content
        if len(content)>15*1024*1024:raise ValueError("15MB以下の画像を選択してください")
        with Image.open(io.BytesIO(bytes(content))) as im:
            widget=result_ui(im.copy(),source="アップロード画像")
        with result_output:
            result_output.clear_output(wait=True)
            display(widget)
    except Exception as e:
        with result_output:
            result_output.clear_output(wait=True)
            display(ui.alert(f"画像の読み込みに失敗しました: {e}",kind="danger"))

mode.observe(set_mode,names="value")
random_button.on_click(run_random)
upload.observe(run_upload,names="value")
set_mode(None)
display(ui.card("トマトの葉の病気を判別","プルダウンでモードを選び、AIの予測結果を確認しよう。"))
display(widgets.VBox([mode,test_panel,upload_panel,result_output]))


## 教材としての注意
学習と検証はtrainだけを使用し、最後のtest評価までtestは学習に使用しません。GitHubのmanifestに記載されている重複除去はSHA-256による**完全一致**のみで、同一の葉を異なる角度で撮った画像などの混入を排除できたことは意味しません。

**ライセンス**：このNotebook自体のサンプルコードとは別に、画像データの再利用・再配布は出典および権利関係を確認してください。GitHubのケース別READMEにはライセンスの確定的な表記がありません。

**対象データ**：v1.1.0（5クラス・7,209枚）。

**今回の学習**：4分割交差検証でOptunaが学習率・Dropout・畳み込みブロック数・フィルタ数を選択。T4で画像読み込みを並列化。CNNにはBatchNormを追加。testは最後だけ使用。
